<a href="https://colab.research.google.com/github/t-snd/g1_manip/blob/worktree-phase6-train/notebooks/train_act_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# フェーズ6：ACT 学習（Colab, 状態のみ・リーチング）

g1_manip で生成したデモ（HF Hub の private データセット）で、LeRobot の ACT を学習する。

**事前準備**
- ランタイム → ランタイムのタイプを変更 → **T4 GPU**
- 左サイドバーの鍵アイコン（シークレット）に **`HF_TOKEN`**（read 権限以上）を登録し、「ノートブックからのアクセス」をオン。
  **トークンをこのノートブックに直接書かないこと**（リポジトリは public）
- 学習を W&B で可視化するなら、同じくシークレットに **`WANDB_API_KEY`**（https://wandb.ai/authorize で取得）を登録し、
  「ノートブックからのアクセス」をオン。使わないなら手順 2 で `USE_WANDB = False` にする

**使い方**
- 初回：手順 1 → 2 → 3 → 4 → 5 → 6（学習）→ 8 → 9
- 切断されたとき：手順 1〜5 を実行してから **手順 7（再開）**。チェックポイントは Drive 上にあるので、最後に保存したところから続く
- 学習・データ取得は `!python ...`（別プロセス）で動かすので、pip のインストール後にセッションを再起動しなくてよい
  （手順 3 の後に「セッションを再起動」ボタンが出ても無視してよい。再起動した場合は手順 2 からやり直す）。
  手順 8 でエラーになったら「ランタイム → セッションを再起動」してから、手順 2・8 を実行する
- 学習が途中で例外で落ちてもセルは成功扱いになることがある（`| tee` のため）。出力の末尾に Traceback が無いか確認する
- W&B を使うと、学習中の loss などを https://wandb.ai のプロジェクト `WANDB_PROJECT` でリアルタイムに見られる。
  再開しても同じ run に続けて記録される（run ID はチェックポイントの `train_config.json` に保存される）

## 1. GPU と Python の確認

In [1]:
import subprocess, sys

print(sys.version)
assert sys.version_info >= (3, 12), "lerobot 0.6.1 は Python 3.12 以上が必要。Colab の Python が古い"

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
print(gpu.stdout or gpu.stderr)
if gpu.returncode != 0:
    raise RuntimeError("GPU が無い。ランタイムのタイプを T4 GPU に変更すること")
if "T4" not in gpu.stdout:
    print("⚠️ 警告: GPU が T4 ではない。AMP の float16 は T4（bfloat16 非対応）に合わせた設定")

3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
Tesla T4, 15360 MiB



## 2. 設定と Google Drive のマウント

- 学習の出力（チェックポイント・ログ）は **Drive 上**に置く（切断されても残る）
- `RUN_NAME` を変えれば別の学習として最初からやり直せる（同じ名前で最初から学習しようとすると手順 6 が止める）

In [2]:
import os
import shutil
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

LEROBOT_VERSION = "0.6.1"                  # ローカル（Mac）と同じ版
DATASET_REPO_ID = "sanada331/g1_reach"     # scripts/05b_push_dataset.py で上げた private データセット
RUN_NAME = "act_g1_reach_v1"

DRIVE_ROOT = Path("/content/drive/MyDrive/g1_act")
RUN_DIR = DRIVE_ROOT / "runs" / RUN_NAME   # lerobot-train の output_dir
LOG_DIR = DRIVE_ROOT / "logs"              # 標準出力のログ（RUN_DIR は lerobot-train が作るので別にする）
LOG_FILE = LOG_DIR / f"{RUN_NAME}.log"
WRAPPER = Path("/content/train_act.py")

# 学習の設定（計画書：状態ベースは 2〜5 万ステップ、batch 32〜64、保存は 2,000〜5,000 ステップごと）
STEPS = 50_000
BATCH_SIZE = 64
SAVE_FREQ = 5_000
LOG_FREQ = 200
NUM_WORKERS = 2   # Colab 無料版は CPU 2 コア

# W&B（学習の可視化）。シークレット WANDB_API_KEY が必要
USE_WANDB = True
WANDB_PROJECT = "g1_act"

LOG_DIR.mkdir(parents=True, exist_ok=True)
print("RUN_DIR:", RUN_DIR, "（既にある）" if RUN_DIR.exists() else "（まだ無い）")

# チェックポイント 1 個 ≈ 460MB（pretrained_model 154MB ＋ training_state 307MB）。50,000 / 5,000 = 10 個で約 4.6GB
need_gb = 0.46 * (STEPS // SAVE_FREQ)
free_gb = shutil.disk_usage("/content/drive").free / 1e9
print(f"Drive の空き {free_gb:.1f}GB / 必要量の目安 {need_gb:.1f}GB")
if free_gb < need_gb * 1.2:
    print("⚠️ Drive の空きが足りないおそれ。SAVE_FREQ を大きくするか、Drive を整理する")


import json


def _safetensors_ok(path: Path) -> bool:
    # ヘッダ（先頭 8 バイトの長さ＋JSON）が読めて、ファイルサイズがヘッダの示すデータ末尾と一致するか
    try:
        with open(path, "rb") as f:
            n = int.from_bytes(f.read(8), "little")
            if n > path.stat().st_size:
                return False
            header = json.loads(f.read(n))
        end = max((v["data_offsets"][1] for k, v in header.items() if k != "__metadata__"), default=0)
        return path.stat().st_size == 8 + n + end
    except Exception:
        return False


def _json_ok(path: Path) -> bool:
    try:
        json.loads(path.read_text())
        return True
    except Exception:
        return False


def checkpoint_complete(ckpt: Path) -> bool:
    # lerobot は training_step.json → rng_state → optimizer_state（約 300MB）→ optimizer_param_groups.json の順に書くので、
    # ファイルの有無だけでなく、safetensors が最後まで書かれているか・JSON が読めるかまで確かめる
    pm, ts = ckpt / "pretrained_model", ckpt / "training_state"
    return (all(_json_ok(p) for p in (pm / "config.json", pm / "train_config.json",
                                      ts / "training_step.json", ts / "optimizer_param_groups.json"))
            and all(_safetensors_ok(p) for p in (pm / "model.safetensors", ts / "optimizer_state.safetensors",
                                                 ts / "rng_state.safetensors")))


def latest_complete_checkpoint(run_dir: Path) -> Path:
    # 保存が最後まで終わっている最新のチェックポイントを返す。不完全なものは理由を表示して飛ばす
    ckpt_root = run_dir / "checkpoints"
    ckpts = sorted((p for p in ckpt_root.iterdir() if p.is_dir() and p.name.isdigit()),
                   key=lambda p: int(p.name), reverse=True) if ckpt_root.is_dir() else []
    for p in ckpts:
        if checkpoint_complete(p):
            return p
        print(f"⚠️ {p.name} は保存が途中で切れている（飛ばす）")
    raise FileNotFoundError(f"{ckpt_root} に完全なチェックポイントが無い。最初から学習し直すには、"
                            f"Drive の {run_dir} を削除するか RUN_NAME を変えてから手順 6 を実行する")


def checkpoint_total_steps(ckpt: Path) -> int:
    # そのチェックポイントを作った学習の総ステップ数（train_config.json の steps）
    return json.loads((ckpt / "pretrained_model" / "train_config.json").read_text())["steps"]

Mounted at /content/drive
RUN_DIR: /content/drive/MyDrive/g1_act/runs/act_g1_reach_v1 （既にある）
Drive の空き 10.5GB / 必要量の目安 4.6GB


## 3. lerobot のインストール（ローカルと同じ版）

Colab に元から入っている `diffusers` は、同じく元から入っている古い `torchao` と合わず import に失敗する
（`cannot import name 'FqnToConfig' from 'torchao.quantization'`）。lerobot は `diffusers` が入っていると
GR00T 用のコードでそれを import するので、ACT でも学習が起動しない。ACT は `diffusers` を使わないのでアンインストールする
（ローカル（Mac）の環境にも入っていない）。

In [3]:
!pip install -q "lerobot[dataset,training]=={LEROBOT_VERSION}"
!pip uninstall -y -q diffusers
!python -c "import lerobot, torch, accelerate, wandb; print('lerobot', lerobot.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| accelerate', accelerate.__version__, '| wandb', wandb.__version__)"
!python -c "import lerobot.scripts.lerobot_train; print('lerobot_train の import OK')"

lerobot 0.6.1 | torch 2.11.0+cu130 | cuda True | accelerate 1.15.0 | wandb 0.27.2
lerobot_train の import OK


## 4. トークン（HF・W&B）を環境変数に設定し、データセットを取得・確認する

Colab のシークレットはノートブックのプロセスからしか読めないので、環境変数 `HF_TOKEN`・`WANDB_API_KEY` に入れて `!python`（別プロセス）へ渡す。

In [4]:
import subprocess
import sys
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")   # 値は表示しない
os.environ["DATASET_REPO_ID"] = DATASET_REPO_ID
print("HF_TOKEN を設定した:", bool(os.environ["HF_TOKEN"]))

if USE_WANDB:
    try:
        os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")   # 値は表示しない
    except Exception as e:
        raise RuntimeError("シークレット WANDB_API_KEY が読めない。登録して「ノートブックからのアクセス」をオンにするか、"
                           "手順 2 で USE_WANDB = False にする") from e
    r = subprocess.run([sys.executable, "-c", "import wandb; print(wandb.Api().viewer.username)"],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("W&B にログインできない（WANDB_API_KEY が無効のおそれ）:\n" + r.stderr[-1000:])
    print("W&B user:", r.stdout.strip())

HF_TOKEN を設定した: True
W&B user: sanada331


In [5]:
%%bash
python - <<'EOF'
import os
from huggingface_hub import HfApi
from lerobot.datasets.lerobot_dataset import LeRobotDataset

print("HF user:", HfApi().whoami()["name"])
m = LeRobotDataset(os.environ["DATASET_REPO_ID"]).meta
print("episodes", m.total_episodes, "| frames", m.total_frames, "| fps", m.fps)
print({k: v["shape"] for k, v in m.features.items()
       if k in ("observation.state", "observation.environment_state", "action")})
print("std min (observation.state):", min(m.stats["observation.state"]["std"]))
EOF

HF user: sanada331
episodes 200 | frames 30000 | fps 50
{'observation.state': (14,), 'observation.environment_state': (3,), 'action': (14,)}
std min (observation.state): 0.01


Fetching 8 files: 100%|██████████| 8/8 [00:00<00:00, 12.13it/s]


期待値：episodes 200 / frames 30000 / fps 50、state 14・environment_state 3・action 14、std の最小値 0.01（`STATS_STD_FLOOR`）。

## 5. 学習ラッパーを書き出す

`lerobot-train` をそのまま使うと、Drive 上で `checkpoints/last` のシンボリックリンク作成に失敗して止まるおそれがある（Drive のマウント先ではシンボリックリンクを作れない場合がある）。
ラッパーで失敗を握りつぶし、代わりに `checkpoints/last_step.txt` に最新ステップを書く。

In [6]:
%%writefile /content/train_act.py
"""lerobot-train（lerobot 0.6.1）を Colab の Google Drive 上で動かすためのラッパー。

1. チェックポイント保存後の `checkpoints/last` シンボリックリンク作成が Drive 上で失敗しても学習を止めない
   （代わりに checkpoints/last_step.txt に最新ステップ名を書く。再開は train_config.json のパスを直接指定するので last は使わない）
2. ログの "step:10K" は丸められているので、正確なステップ数 "step_exact:<n>" を先頭に付ける（loss 曲線用）
"""
from pathlib import Path

import lerobot.scripts.lerobot_train as lt
from lerobot.utils import logging_utils

_update_last_checkpoint = lt.update_last_checkpoint


def update_last_checkpoint(checkpoint_dir):
    try:
        _update_last_checkpoint(checkpoint_dir)
    except OSError as e:
        print(f"[train_act] last シンボリックリンクを作れなかった（{e}）。last_step.txt に記録する")
    (Path(checkpoint_dir).parent / "last_step.txt").write_text(Path(checkpoint_dir).name)


lt.update_last_checkpoint = update_last_checkpoint

_tracker_str = logging_utils.MetricsTracker.__str__
logging_utils.MetricsTracker.__str__ = lambda self: f"step_exact:{self.steps} " + _tracker_str(self)

if __name__ == "__main__":
    lt.main()

Writing /content/train_act.py


## 6. 学習（最初から）

- `policy.use_amp=true`：CUDA の autocast の既定は **float16**（T4 は bfloat16 非対応なので、これでよい）
- `policy.push_to_hub=false`：学習済みモデルは Hub に上げない（Drive に置く）
- 学習中のログは Drive の `LOG_FILE` にも追記される
- W&B：ログの `Track this run -->` の URL を開くと、loss などのグラフが `LOG_FREQ` ステップごとに更新される。
  `wandb.disable_artifact=true` で、チェックポイントは W&B に上げない（Drive にある。上げると保存のたびに約 150MB）
- T4 での所要時間はログの `updt_s`（1 ステップの時間）× 残りステップで見積もる。12 時間を超えそうなら STEPS を減らす

In [ ]:
if RUN_DIR.exists():
    raise FileExistsError(f"{RUN_DIR} が既にある。続きからなら手順 7（再開）を、最初からなら RUN_NAME を変える")
if LOG_FILE.exists():   # 同じ RUN_NAME でやり直すとき、古いログが loss 曲線に混ざらないよう退避する
    LOG_FILE.rename(LOG_FILE.with_name(f"{LOG_FILE.stem}.old{int(LOG_FILE.stat().st_mtime)}.log"))
WANDB_ENABLE = "true" if USE_WANDB else "false"

!python {WRAPPER} \
    --dataset.repo_id={DATASET_REPO_ID} \
    --policy.type=act \
    --policy.device=cuda \
    --policy.use_amp=true \
    --policy.push_to_hub=false \
    --output_dir={RUN_DIR} \
    --job_name={RUN_NAME} \
    --steps={STEPS} \
    --batch_size={BATCH_SIZE} \
    --save_freq={SAVE_FREQ} \
    --log_freq={LOG_FREQ} \
    --num_workers={NUM_WORKERS} \
    --wandb.enable={WANDB_ENABLE} \
    --wandb.project={WANDB_PROJECT} \
    --wandb.disable_artifact=true \
    2>&1 | tee -a {LOG_FILE}

INFO 2026-10-23 13:30:23 ot_train.py:272 {'batch_size': 64,
 'checkpoint_path': None,
 'cudnn_deterministic': False,
 'dataloader_multiprocessing_context': 'spawn',
 'dataset': {'depth_output_unit': 'mm',
             'episodes': None,
             'eval_split': 0.0,
             'image_transforms': {'enable': False,
                                  'max_num_transforms': 3,
                                  'random_order': False,
                                  'tfs': {'affine': {'kwargs': {'degrees': [-5.0,
                                                                            5.0],
                                                                'translate': [0.05,
                                                                              0.05]},
                                                     'type': 'RandomAffine',
                                                     'weight': 1.0},
                                          'brightness': {'kwargs': {'brightness': [0.

## 7. 再開（切断された後）

Drive 上で、保存が最後まで終わっている最新のチェックポイントを探し（保存の途中で切れたものは飛ばす）、
その `train_config.json` を指定して `--resume=true` で続きから学習する。設定（ステップ数など）はチェックポイントのものが使われる。

W&B の設定もチェックポイントのものが使われる。`train_config.json` に run ID が入っているので、同じ run に続けて記録される
（lerobot は run ID が無いと `wandb/latest-run` のシンボリックリンクから探すが、Drive 上では作れないことがあるので、それには頼らない）。
ただし、最後の保存から切断までの区間は W&B 側に切断前の値が残り、再開後の同じステップの値は記録されない
（W&B は step が戻るログを捨てる）。その区間は手順 8 の曲線（再開後の値を採用）と食い違うことがある。

In [ ]:
ckpt = latest_complete_checkpoint(RUN_DIR)
total = checkpoint_total_steps(ckpt)
print("再開するチェックポイント:", ckpt, f"（ステップ {int(ckpt.name)} / {total}）")
wandb_cfg = json.loads((ckpt / "pretrained_model" / "train_config.json").read_text())["wandb"]
print("W&B:", f"有効（project {wandb_cfg['project']}, run ID {wandb_cfg['run_id']}）" if wandb_cfg["enable"] else "無効")
if wandb_cfg["enable"] and not os.environ.get("WANDB_API_KEY"):
    raise RuntimeError("この学習は W&B が有効なのに WANDB_API_KEY が無い。USE_WANDB = True にして手順 2・4 を実行する")
if int(ckpt.name) >= total:
    print("⚠️ 既に最終ステップまで学習済み。手順 8・9 へ")
else:
    !python {WRAPPER} \
        --config_path={ckpt}/pretrained_model/train_config.json \
        --resume=true \
        2>&1 | tee -a {LOG_FILE}

## 8. 学習曲線（loss）

In [7]:
import re
import matplotlib.pyplot as plt

text = LOG_FILE.read_text(errors="ignore")
num = r"([-\w.+]+)"   # 数値のほか nan / inf も拾う
pat = re.compile(rf"step_exact:(\d+) .*? loss:{num} .*? l1_loss:{num} kld_loss:{num}")
n_logged = len(re.findall(r"step_exact:\d+", text))
rows = {}
for m in pat.finditer(text):
    rows[int(m.group(1))] = tuple(float(x) for x in m.groups()[1:])   # 再開で同じステップが重複したら後のものを使う
steps = sorted(rows)
print(f"{len(steps)} 点（ログ行 {n_logged}）, 最終ステップ {steps[-1] if steps else None}")
bad = [s for s in steps if any(v != v or abs(v) == float("inf") for v in rows[s])]
if bad:
    print(f"⚠️ nan / inf のステップがある（fp16 で発散したおそれ）: {bad[:10]}")
if steps:
    loss, l1, kld = zip(*(rows[s] for s in steps))
    print(f"最後の値: loss={loss[-1]:.4f}, l1_loss={l1[-1]:.4f}, kld_loss={kld[-1]:.4f}")
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, y, name in zip(axes, (loss, l1, kld), ("loss (= l1 + kl_weight * kld)", "l1_loss", "kld_loss")):
        ax.plot(steps, y)
        if all(v > 0 for v in y):   # ログは小数 3 桁なので、kld が 0.000 になると対数軸に描けない
            ax.set_yscale("log")
        ax.set_xlabel("step")
        ax.set_title(name)
        ax.grid(True, which="both", alpha=0.3)
    plt.tight_layout()
    plt.show()

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/g1_act/logs/act_g1_reach_v1.log'

## 9. 最終チェックポイントを Mac へ持っていく

最終チェックポイントの `pretrained_model` フォルダを zip にして Drive（`g1_act/exports/`）に置き、ブラウザでダウンロードもする。

Mac 側では、g1_manip の `outputs/<RUN_NAME>/pretrained_model/` に展開する（`outputs/` は git 管理外）：

```bash
cd g1_manip
mkdir -p outputs/act_g1_reach_v1
unzip ~/Downloads/act_g1_reach_v1_pretrained_model.zip -d outputs/act_g1_reach_v1/
ls outputs/act_g1_reach_v1/pretrained_model   # config.json, model.safetensors, train_config.json, policy_*processor* など
```

In [8]:
import shutil
from google.colab import files

ckpt = latest_complete_checkpoint(RUN_DIR)
total = checkpoint_total_steps(ckpt)
print("最終チェックポイント:", ckpt, f"（ステップ {int(ckpt.name)} / {total}）")
if int(ckpt.name) < total:
    print("⚠️ まだ最終ステップまで学習していない")

export_dir = DRIVE_ROOT / "exports"
export_dir.mkdir(parents=True, exist_ok=True)
zip_path = shutil.make_archive(str(export_dir / f"{RUN_NAME}_pretrained_model"), "zip",
                               root_dir=ckpt, base_dir="pretrained_model")
print("Drive に保存:", zip_path)
!unzip -l "{zip_path}"
files.download(zip_path)

最終チェックポイント: /content/drive/MyDrive/g1_act/runs/act_g1_reach_v1/checkpoints/040000 （ステップ 40000 / 50000）
⚠️ まだ最終ステップまで学習していない
Drive に保存: /content/drive/MyDrive/g1_act/exports/act_g1_reach_v1_pretrained_model.zip
Archive:  /content/drive/MyDrive/g1_act/exports/act_g1_reach_v1_pretrained_model.zip
  Length      Date    Time    Name
---------  ---------- -----   ----
        0  2026-10-23 19:35   pretrained_model/
     1487  2026-10-23 19:35   pretrained_model/config.json
160970000  2026-10-23 19:35   pretrained_model/model.safetensors
     5882  2026-10-23 19:35   pretrained_model/train_config.json
     7224  2026-10-23 19:35   pretrained_model/policy_preprocessor_step_3_normalizer_processor.safetensors
     1120  2026-10-23 19:35   pretrained_model/policy_preprocessor.json
     7224  2026-10-23 19:35   pretrained_model/policy_postprocessor_step_0_unnormalizer_processor.safetensors
      661  2026-10-23 19:35   pretrained_model/policy_postprocessor.json
---------                     ------

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>